In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

1 — Install & Imports

In [2]:
!pip install transformers datasets scikit-learn -q

import pandas as pd
import numpy as np
import re
import torch
from datasets import load_dataset
from sklearn.utils.class_weight import compute_class_weight

def minimal_clean(text):
    text = str(text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU!'}")

GPU: Tesla P100-PCIE-16GB


2 — Load All Datasets

In [3]:
print("Loading datasets...")

# Depressed
ds_adh = load_dataset("adhammai/depression_clean", split="train")
df_adh = ds_adh.to_pandas()

ds_j1  = load_dataset("Jillian/Depression_detection_reddit", split="train")
df_j1  = ds_j1.to_pandas()

ds_j2  = load_dataset("Jillian/depression_detection_test", split="test")
df_j2  = ds_j2.to_pandas()

# Non-depressed boost
ds_emo = load_dataset("google-research-datasets/go_emotions", "simplified", split="train")
df_emo = ds_emo.to_pandas()

# Severity (calibration only)
ds_sev = load_dataset("siyangliu/Depression_Severity_Dataset", split="train")
df_sev = ds_sev.to_pandas()

print("✅ All loaded!")

Loading datasets...


README.md:   0%|          | 0.00/432 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/217k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/56.7k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

...

Reddit_depression_dataset_test.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/2842 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/355 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/356 [00:00<?, ? examples/s]

✅ All loaded!


3 — Build Binary Dataset

In [4]:
# Depressed samples
dep_adh = pd.DataFrame({
    "text"  : df_adh[df_adh["is_depression"]==1]["clean_text"].apply(minimal_clean),
    "label" : 1, "source": "adhammai"
})

dep_j1 = pd.DataFrame({
    "text"  : df_j1[df_j1["output"]=="depressed"]["input"].apply(minimal_clean),
    "label" : 1, "source": "jillian_reddit"
})

dep_j2 = pd.DataFrame({
    "text"  : df_j2[df_j2["output"]=="depressed"]["input"].apply(minimal_clean),
    "label" : 1, "source": "jillian_test"
})

# Non-depressed samples
nodep_adh = pd.DataFrame({
    "text"  : df_adh[df_adh["is_depression"]==0]["clean_text"].apply(minimal_clean),
    "label" : 0, "source": "adhammai"
})

nodep_j1 = pd.DataFrame({
    "text"  : df_j1[df_j1["output"]=="normal"]["input"].apply(minimal_clean),
    "label" : 0, "source": "jillian_reddit"
})

nodep_j2 = pd.DataFrame({
    "text"  : df_j2[df_j2["output"]=="normal"]["input"].apply(minimal_clean),
    "label" : 0, "source": "jillian_test"
})

# go-emotions: positive/neutral Reddit comments
POSITIVE_LABELS = {0,1,2,3,4,5,13,15,17,18,20,21,23,27}
df_emo["label_int"] = df_emo["labels"].apply(lambda x: x[0] if len(x)==1 else -1)
df_emo = df_emo[df_emo["label_int"] != -1]
go_nodep = df_emo[df_emo["label_int"].isin(POSITIVE_LABELS)][["text"]].copy()
go_nodep["label"]  = 0
go_nodep["source"] = "go_emotions"

# Combine
final_df = pd.concat([
    dep_adh, dep_j1, dep_j2,
    nodep_adh, nodep_j1, nodep_j2,
    go_nodep
], ignore_index=True)

final_df = final_df.dropna(subset=["text","label"])
final_df = final_df[final_df["text"].str.len() > 20]
final_df = final_df[final_df["label"].isin([0,1])]
final_df = final_df.drop_duplicates(subset=["text"])
final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("="*60)
print("BINARY TRAINING DATA")
print("="*60)
print(f"Total         : {len(final_df)}")
print(f"Depressed     : {(final_df['label']==1).sum()}")
print(f"Not depressed : {(final_df['label']==0).sum()}")
print(f"Balance       : {final_df['label'].mean():.1%} depressed")
print(f"\nSource breakdown:")
print(final_df.groupby(["source","label"]).size().unstack(fill_value=0))
print(f"\nAvg length depressed    : {final_df[final_df['label']==1]['text'].str.len().mean():.0f} chars")
print(f"Avg length not depressed: {final_df[final_df['label']==0]['text'].str.len().mean():.0f} chars")

BINARY TRAINING DATA
Total         : 32035
Depressed     : 3722
Not depressed : 28313
Balance       : 11.6% depressed

Source breakdown:
label               0     1
source                     
adhammai          453   447
go_emotions     24659     0
jillian_reddit   2581  2586
jillian_test      620   689

Avg length depressed    : 664 chars
Avg length not depressed: 73 chars


4 — Severity Calibration Data

In [5]:
df_sev["text"] = df_sev["text"].apply(minimal_clean)
df_sev = df_sev[df_sev["text"].str.len() > 30].reset_index(drop=True)
df_sev["severity_score"] = df_sev["label"].map({
    "minimum": 0, "mild": 1, "moderate": 2, "severe": 3
})

print("="*60)
print("SEVERITY CALIBRATION DATA")
print("="*60)
print(f"Total    : {len(df_sev)}")
print(f"Severity : {df_sev['label'].value_counts().to_dict()}")

# Class weights for training
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0,1]),
    y=final_df["label"].values
)
print(f"\nClass weights:")
print(f"  Not depressed (0): {weights[0]:.4f}")
print(f"  Depressed     (1): {weights[1]:.4f}")
print(f"\n✅ Ready to train!")
print(f"   Binary   → {len(final_df)} samples")
print(f"   Severity → {len(df_sev)} samples")

SEVERITY CALIBRATION DATA
Total    : 2838
Severity : {'minimum': 2059, 'moderate': 323, 'mild': 232, 'severe': 224}

Class weights:
  Not depressed (0): 0.5657
  Depressed     (1): 4.3035

✅ Ready to train!
   Binary   → 32035 samples
   Severity → 2838 samples


 5 — Model Definition

In [32]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Load token from Kaggle secrets
secrets    = UserSecretsClient()
hf_token   = secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("✅ HuggingFace login successful!")

# Now load model
MODEL_NAME = "mental/mental-roberta-base"
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


✅ HuggingFace login successful!
Device: cuda


In [7]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score, classification_report
import numpy as np

MODEL_NAME = "mental/mental-roberta-base"
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

class MentalHealthDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length      = self.max_len,
            padding         = "max_length",
            truncation      = True,
            return_tensors  = "pt"
        )
        return {
            "input_ids"     : enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "label"         : torch.tensor(self.labels[idx], dtype=torch.long)
        }

class MentalBERTClassifier(nn.Module):
    def __init__(self, model_name, num_classes=2, dropout=0.3):
        super().__init__()
        self.bert    = AutoModel.from_pretrained(model_name)
        hidden_size  = self.bert.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        out  = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls  = out.last_hidden_state[:, 0, :]  # [CLS] token
        cls  = self.dropout(cls)
        return self.classifier(cls)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("✅ Tokenizer loaded!")

Device: cuda


config.json:   0%|          | 0.00/682 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/327 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

✅ Tokenizer loaded!


In [8]:
import os

# ── Find exact paths first ─────────────────────────────────────────
BASE = "/kaggle/input/datasets/gaurkumarsoni/depression-mentalbert-models"
print("Files found:")
for root, dirs, files in os.walk(BASE):
    for file in files:
        path = os.path.join(root, file)
        size = os.path.getsize(path)/1024/1024
        print(f"  ✅ {path} ({size:.1f} MB)")

# ── Load binary model (no subfolder) ──────────────────────────────
binary_model_final = MentalBERTClassifier(MODEL_NAME).to(DEVICE)
binary_model_final.load_state_dict(
    torch.load(
        f"{BASE}/model_weights.pt",
        map_location=DEVICE
    )
)
binary_model_final.eval()
print("\n✅ Binary model loaded!")

# ── Severity model ─────────────────────────────────────────────────
class SeverityClassifier3(nn.Module):
    def __init__(self, model_name, num_classes=3, dropout=0.3):
        super().__init__()
        self.bert    = AutoModel.from_pretrained(model_name)
        hidden_size  = self.bert.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        cls = self.dropout(cls)
        return self.classifier(cls)

sev_model_final = SeverityClassifier3(MODEL_NAME).to(DEVICE)
sev_model_final.load_state_dict(
    torch.load(
        f"{BASE}/model_weights_3class.pt",
        map_location=DEVICE
    )
)
sev_model_final.eval()
print("✅ Severity model loaded!")

# ── Tokenizer ──────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(BASE)
print("✅ Tokenizer loaded!")

# Calibration
BINARY_THRESHOLD = 0.6769
TEMPERATURE      = 0.1461
print(f"\n✅ Threshold  : {BINARY_THRESHOLD}")
print(f"✅ Temperature: {TEMPERATURE}")
print("\n🚀 All models ready!")

Files found:
  ✅ /kaggle/input/datasets/gaurkumarsoni/depression-mentalbert-models/tokenizer.json (3.4 MB)
  ✅ /kaggle/input/datasets/gaurkumarsoni/depression-mentalbert-models/tokenizer_config.json (0.0 MB)
  ✅ /kaggle/input/datasets/gaurkumarsoni/depression-mentalbert-models/model_weights_3class.pt (476.3 MB)
  ✅ /kaggle/input/datasets/gaurkumarsoni/depression-mentalbert-models/model_weights.pt (476.3 MB)


pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: mental/mental-roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]


✅ Binary model loaded!


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: mental/mental-roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Severity model loaded!
✅ Tokenizer loaded!

✅ Threshold  : 0.6769
✅ Temperature: 0.1461

🚀 All models ready!


6 — Training with Cross-Validation

In [ ]:
# ── Config ─────────────────────────────────────────────────────────
EPOCHS      = 3
BATCH_SIZE  = 32
LR          = 2e-5
MAX_LEN     = 128
N_FOLDS     = 3
WEIGHT_DECAY= 0.01

class_weights_tensor = torch.tensor(weights, dtype=torch.float).to(DEVICE)

# Full dataset — no subsampling
texts_arr  = np.array(final_df["text"].tolist())
labels_arr = np.array(final_df["label"].astype(int).tolist())

print(f"Training on  : {len(texts_arr)} samples")
print(f"Depressed    : {(labels_arr==1).sum()}")
print(f"Not depressed: {(labels_arr==0).sum()}")
print(f"Max length   : {MAX_LEN} tokens")
print(f"Batch size   : {BATCH_SIZE}")
print(f"Folds        : {N_FOLDS}")
print(f"Epochs       : {EPOCHS}")

# ── Cross Validation ───────────────────────────────────────────────
skf          = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fold_results = []
best_overall_f1    = 0
best_overall_model = None

for fold, (train_idx, val_idx) in enumerate(skf.split(texts_arr, labels_arr)):
    print(f"\n{'='*60}")
    print(f"FOLD {fold+1}/{N_FOLDS}")
    print(f"{'='*60}")

    train_texts  = texts_arr[train_idx].tolist()
    train_labels = labels_arr[train_idx].tolist()
    val_texts    = texts_arr[val_idx].tolist()
    val_labels   = labels_arr[val_idx].tolist()

    train_ds = MentalHealthDataset(train_texts, train_labels, tokenizer, MAX_LEN)
    val_ds   = MentalHealthDataset(val_texts,   val_labels,   tokenizer, MAX_LEN)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
    val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

    model     = MentalBERTClassifier(MODEL_NAME).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=0.1)
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps = len(train_dl) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps   = int(0.1 * total_steps),
        num_training_steps = total_steps
    )

    best_val_f1    = 0
    best_val_model = None
    patience_count = 0

    for epoch in range(EPOCHS):
        train_loss, train_f1 = train_epoch(
            model, train_dl, optimizer, scheduler, criterion
        )
        val_loss, val_f1, val_auc, _, _, _ = eval_epoch(
            model, val_dl, criterion
        )

        print(f"  Epoch {epoch+1}: "
              f"train_loss={train_loss:.4f} train_f1={train_f1:.4f} | "
              f"val_loss={val_loss:.4f} val_f1={val_f1:.4f} val_auc={val_auc:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1    = val_f1
            best_val_model = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= 2:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    fold_results.append(best_val_f1)
    print(f"  Best F1 fold {fold+1}: {best_val_f1:.4f}")

    if best_val_f1 > best_overall_f1:
        best_overall_f1    = best_val_f1
        best_overall_model = best_val_model

    # Free GPU memory between folds
    del model
    torch.cuda.empty_cache()

print(f"\n{'='*60}")
print(f"CROSS-VALIDATION RESULTS")
print(f"{'='*60}")
print(f"Fold F1s : {[f'{f:.4f}' for f in fold_results]}")
print(f"Mean F1  : {np.mean(fold_results):.4f} ± {np.std(fold_results):.4f}")
print(f"Best F1  : {best_overall_f1:.4f}")


7 — Save Binary Model

In [ ]:
import os
os.makedirs("mentalbert_binary", exist_ok=True)

# Load best weights
model = MentalBERTClassifier(MODEL_NAME).to(DEVICE)
model.load_state_dict({k: v.to(DEVICE) for k, v in best_overall_model.items()})
model.eval()

# Save
torch.save(best_overall_model, "mentalbert_binary/model_weights.pt")
tokenizer.save_pretrained("mentalbert_binary")
print(f"✅ Binary model saved! Best F1: {best_overall_f1:.4f}")

8 — Severity Calibration

In [ ]:
# ── Train severity head on siyangliu dataset ───────────────────────
SEVERITY_MAP = {"minimum": 0, "mild": 1, "moderate": 2, "severe": 3}
df_sev["severity_score"] = df_sev["label"].map(SEVERITY_MAP)

sev_texts  = df_sev["text"].tolist()
sev_labels = df_sev["severity_score"].tolist()

sev_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0,1,2,3]),
    y=df_sev["severity_score"].values
)
sev_weights_tensor = torch.tensor(sev_weights, dtype=torch.float).to(DEVICE)
print(f"Severity class weights: {sev_weights.round(3)}")

class SeverityClassifier(nn.Module):
    def __init__(self, model_name, num_classes=4, dropout=0.3):
        super().__init__()
        self.bert    = AutoModel.from_pretrained(model_name)
        hidden_size  = self.bert.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        cls = self.dropout(cls)
        return self.classifier(cls)

# Use 80/20 split for severity (smaller dataset)
from sklearn.model_selection import train_test_split
s_train_texts, s_val_texts, s_train_labels, s_val_labels = train_test_split(
    sev_texts, sev_labels, test_size=0.2,
    stratify=sev_labels, random_state=42
)

s_train_ds = MentalHealthDataset(s_train_texts, s_train_labels, tokenizer, MAX_LEN)
s_val_ds   = MentalHealthDataset(s_val_texts,   s_val_labels,   tokenizer, MAX_LEN)
s_train_dl = DataLoader(s_train_ds, batch_size=16, shuffle=True,  num_workers=2)
s_val_dl   = DataLoader(s_val_ds,   batch_size=16, shuffle=False, num_workers=2)

sev_model     = SeverityClassifier(MODEL_NAME).to(DEVICE)
sev_criterion = nn.CrossEntropyLoss(weight=sev_weights_tensor, label_smoothing=0.1)
sev_optimizer = AdamW(sev_model.parameters(), lr=1e-5, weight_decay=0.01)
sev_steps     = len(s_train_dl) * 4
sev_scheduler = get_linear_schedule_with_warmup(
    sev_optimizer,
    num_warmup_steps   = int(0.1 * sev_steps),
    num_training_steps = sev_steps
)

best_sev_f1    = 0
best_sev_model = None

for epoch in range(4):
    # Train
    sev_model.train()
    train_loss, preds, trues = 0, [], []
    for batch in s_train_dl:
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        lbl  = batch["label"].to(DEVICE)
        sev_optimizer.zero_grad()
        logits = sev_model(ids, mask)
        loss   = sev_criterion(logits, lbl)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(sev_model.parameters(), 1.0)
        sev_optimizer.step()
        sev_scheduler.step()
        train_loss += loss.item()
        preds.extend(torch.argmax(logits,1).cpu().numpy())
        trues.extend(lbl.cpu().numpy())
    train_f1 = f1_score(trues, preds, average="macro")

    # Eval
    sev_model.eval()
    val_loss, val_preds, val_trues = 0, [], []
    with torch.no_grad():
        for batch in s_val_dl:
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            lbl  = batch["label"].to(DEVICE)
            logits = sev_model(ids, mask)
            loss   = sev_criterion(logits, lbl)
            val_loss += loss.item()
            val_preds.extend(torch.argmax(logits,1).cpu().numpy())
            val_trues.extend(lbl.cpu().numpy())
    val_f1 = f1_score(val_trues, val_preds, average="macro")

    print(f"Epoch {epoch+1}: train_f1={train_f1:.4f} | val_f1={val_f1:.4f}")

    if val_f1 > best_sev_f1:
        best_sev_f1    = val_f1
        best_sev_model = {k: v.cpu().clone() for k, v in sev_model.state_dict().items()}

print(f"\n✅ Best severity F1: {best_sev_f1:.4f}")

os.makedirs("mentalbert_severity", exist_ok=True)
torch.save(best_sev_model, "mentalbert_severity/model_weights.pt")
print("✅ Severity model saved!")

In [ ]:
# ── Severity Dataset Analysis ──────────────────────────────────────
print("SEVERITY IMBALANCE DIAGNOSIS")
print("="*60)
print(f"Total    : {len(df_sev)}")
print(f"\nClass distribution:")
for label, count in df_sev["label"].value_counts().items():
    pct = count/len(df_sev)*100
    bar = "█" * int(pct/2)
    print(f"  {label:10s}: {count:4d} ({pct:5.1f}%) {bar}")

print(f"\nAvg length by severity:")
for label in ["minimum","mild","moderate","severe"]:
    avg = df_sev[df_sev["label"]==label]["text"].str.len().mean()
    print(f"  {label:10s}: {avg:.0f} chars")

print(f"\nSample minimum (should NOT be depressed):")
print(df_sev[df_sev["label"]=="minimum"]["text"].iloc[0][:150])
print(df_sev[df_sev["label"]=="minimum"]["text"].iloc[1][:150])
print(df_sev[df_sev["label"]=="minimum"]["text"].iloc[2][:150])

print(f"\nSample severe:")
print(df_sev[df_sev["label"]=="severe"]["text"].iloc[0][:150])
print(df_sev[df_sev["label"]=="severe"]["text"].iloc[1][:150])


In [ ]:
# ══════════════════════════════════════════════════════════════════
# IMPROVED SEVERITY TRAINING
# ══════════════════════════════════════════════════════════════════
import random
from torch.utils.data import WeightedRandomSampler

# ── Fix 1: Undersample minimum to reduce dominance ─────────────────
df_min  = df_sev[df_sev["label"]=="minimum"].sample(500, random_state=42)
df_mild = df_sev[df_sev["label"]=="mild"]
df_mod  = df_sev[df_sev["label"]=="moderate"]
df_sev2 = df_sev[df_sev["label"]=="severe"]

# ── Fix 2: Oversample minority classes with augmentation ───────────
def augment_text(text, n=2):
    """Simple augmentation: random word dropout + shuffle sentences"""
    augmented = []
    words = text.split()
    for _ in range(n):
        # Random word dropout (drop 10% of words)
        kept   = [w for w in words if random.random() > 0.1]
        augmented.append(" ".join(kept))
    return augmented

# Augment mild, moderate, severe to ~500 each
def augment_df(df, target=500, label=None):
    current = len(df)
    if current >= target:
        return df.sample(target, random_state=42)
    needed  = target - current
    texts   = df["text"].tolist()
    new_rows = []
    while len(new_rows) < needed:
        for text in texts:
            if len(new_rows) >= needed:
                break
            aug = augment_text(text, n=1)[0]
            new_rows.append({
                "text"          : aug,
                "label"         : label,
                "severity_score": df["severity_score"].iloc[0]
            })
    aug_df = pd.DataFrame(new_rows)
    return pd.concat([df, aug_df], ignore_index=True)

df_mild_aug = augment_df(df_mild, target=500, label="mild")
df_mod_aug  = augment_df(df_mod,  target=500, label="moderate")
df_sev_aug  = augment_df(df_sev2, target=500, label="severe")

# Combine balanced severity dataset
df_sev_balanced = pd.concat([
    df_min, df_mild_aug, df_mod_aug, df_sev_aug
], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

print("BALANCED SEVERITY DATASET")
print("="*60)
print(f"Total    : {len(df_sev_balanced)}")
print(f"Distribution:")
for label in ["minimum","mild","moderate","severe"]:
    count = (df_sev_balanced["label"]==label).sum()
    print(f"  {label:10s}: {count}")

# ── Fix 3: Initialize from our trained binary model ────────────────
print("\nInitializing severity model from binary model weights...")

class SeverityClassifier(nn.Module):
    def __init__(self, model_name, num_classes=4, dropout=0.3):
        super().__init__()
        self.bert    = AutoModel.from_pretrained(model_name)
        hidden_size  = self.bert.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        cls = self.dropout(cls)
        return self.classifier(cls)

sev_model = SeverityClassifier(MODEL_NAME).to(DEVICE)

# Load BERT weights from our trained binary model (transfer learning)
binary_bert_state = {
    k.replace("bert.", ""): v
    for k, v in best_overall_model.items()
    if k.startswith("bert.")
}
sev_model.bert.load_state_dict(binary_bert_state, strict=False)
print("✅ Loaded binary model BERT weights into severity model!")

# ── Fix 4: Stronger class weights ─────────────────────────────────
sev_class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0,1,2,3]),
    y=df_sev_balanced["severity_score"].values
)
# Boost minority classes further
sev_class_weights[1] *= 1.5  # mild
sev_class_weights[2] *= 1.5  # moderate
sev_class_weights[3] *= 2.0  # severe (rarest, most important)

print(f"\nClass weights (boosted):")
for i, (name, w) in enumerate(zip(
    ["minimum","mild","moderate","severe"], sev_class_weights
)):
    print(f"  {name:10s}: {w:.3f}")

sev_weights_tensor = torch.tensor(
    sev_class_weights, dtype=torch.float
).to(DEVICE)

# ── Fix 5: Train/val split ─────────────────────────────────────────
sev_texts  = df_sev_balanced["text"].tolist()
sev_labels = df_sev_balanced["severity_score"].tolist()

s_train_texts, s_val_texts, s_train_labels, s_val_labels = train_test_split(
    sev_texts, sev_labels,
    test_size=0.2, stratify=sev_labels, random_state=42
)

s_train_ds = MentalHealthDataset(s_train_texts, s_train_labels, tokenizer, MAX_LEN)
s_val_ds   = MentalHealthDataset(s_val_texts,   s_val_labels,   tokenizer, MAX_LEN)
s_train_dl = DataLoader(s_train_ds, batch_size=16, shuffle=True,  num_workers=2)
s_val_dl   = DataLoader(s_val_ds,   batch_size=16, shuffle=False, num_workers=2)

# ── Fix 6: More epochs, lower LR, differential LR ─────────────────
# Lower LR for BERT (pretrained), higher for classifier (new)
optimizer_grouped = [
    {"params": sev_model.bert.parameters(),       "lr": 5e-6},
    {"params": sev_model.classifier.parameters(), "lr": 1e-4}
]
sev_optimizer = AdamW(optimizer_grouped, weight_decay=0.01)
sev_steps     = len(s_train_dl) * 6
sev_scheduler = get_linear_schedule_with_warmup(
    sev_optimizer,
    num_warmup_steps   = int(0.15 * sev_steps),
    num_training_steps = sev_steps
)
sev_criterion = nn.CrossEntropyLoss(
    weight=sev_weights_tensor, label_smoothing=0.05
)

# ── Train ──────────────────────────────────────────────────────────
best_sev_f1    = 0
best_sev_model = None
SEVERITY_EPOCHS = 6

print(f"\nTraining severity model for {SEVERITY_EPOCHS} epochs...")
print("="*60)

for epoch in range(SEVERITY_EPOCHS):
    # Train
    sev_model.train()
    train_loss, preds, trues = 0, [], []
    for batch in s_train_dl:
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        lbl  = batch["label"].to(DEVICE)
        sev_optimizer.zero_grad()
        logits = sev_model(ids, mask)
        loss   = sev_criterion(logits, lbl)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(sev_model.parameters(), 1.0)
        sev_optimizer.step()
        sev_scheduler.step()
        train_loss += loss.item()
        preds.extend(torch.argmax(logits,1).cpu().numpy())
        trues.extend(lbl.cpu().numpy())
    train_f1 = f1_score(trues, preds, average="macro")

    # Eval
    sev_model.eval()
    val_preds, val_trues = [], []
    with torch.no_grad():
        for batch in s_val_dl:
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            lbl  = batch["label"].to(DEVICE)
            logits = sev_model(ids, mask)
            val_preds.extend(torch.argmax(logits,1).cpu().numpy())
            val_trues.extend(lbl.cpu().numpy())

    val_f1 = f1_score(val_trues, val_preds, average="macro")
    report = classification_report(
        val_trues, val_preds,
        target_names=["minimum","mild","moderate","severe"],
        output_dict=True
    )

    print(f"Epoch {epoch+1}: train_f1={train_f1:.4f} | val_f1={val_f1:.4f}")
    print(f"  minimum={report['minimum']['f1-score']:.3f} "
          f"mild={report['mild']['f1-score']:.3f} "
          f"moderate={report['moderate']['f1-score']:.3f} "
          f"severe={report['severe']['f1-score']:.3f}")

    if val_f1 > best_sev_f1:
        best_sev_f1    = val_f1
        best_sev_model = {k: v.cpu().clone() for k, v in sev_model.state_dict().items()}

print(f"\n{'='*60}")
print(f"✅ Best severity F1: {best_sev_f1:.4f}")
print(f"   Previous        : 0.4473")
print(f"   Improvement     : +{best_sev_f1-0.4473:.4f}")

os.makedirs("mentalbert_severity", exist_ok=True)
torch.save(best_sev_model, "mentalbert_severity/model_weights.pt")
print("✅ Improved severity model saved!")

Improved

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 3-CLASS SEVERITY TRAINING
# ══════════════════════════════════════════════════════════════════

# ── Map to 3 classes ───────────────────────────────────────────────
df_sev["severity_3class"] = df_sev["label"].map({
    "minimum" : 0,
    "mild"    : 0,
    "moderate": 1,
    "severe"  : 2
})

print("3-Class distribution:")
for i, name in enumerate(["Low (min+mild)", "Moderate", "Severe"]):
    count = (df_sev["severity_3class"]==i).sum()
    pct   = count/len(df_sev)*100
    bar   = "█" * int(pct/3)
    print(f"  {name:20s}: {count:4d} ({pct:5.1f}%) {bar}")

# ── Balance dataset ────────────────────────────────────────────────
df_low  = df_sev[df_sev["severity_3class"]==0].sample(600, random_state=42)
df_mod  = df_sev[df_sev["severity_3class"]==1]
df_high = df_sev[df_sev["severity_3class"]==2]

# Augment moderate and severe to 600 each
def augment_df_3class(df, target=600, class_id=None):
    current = len(df)
    if current >= target:
        return df.sample(target, random_state=42)
    needed   = target - current
    texts    = df["text"].tolist()
    new_rows = []
    while len(new_rows) < needed:
        for text in texts:
            if len(new_rows) >= needed:
                break
            words = text.split()
            kept  = [w for w in words if random.random() > 0.1]
            new_rows.append({
                "text"           : " ".join(kept),
                "label"          : df["label"].iloc[0],
                "severity_score" : df["severity_score"].iloc[0],
                "severity_3class": class_id
            })
    aug_df = pd.DataFrame(new_rows)
    return pd.concat([df, aug_df], ignore_index=True).sample(target, random_state=42)

df_mod_aug  = augment_df_3class(df_mod,  target=600, class_id=1)
df_high_aug = augment_df_3class(df_high, target=600, class_id=2)

df_sev_3class = pd.concat(
    [df_low, df_mod_aug, df_high_aug],
    ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nBalanced 3-class dataset:")
for i, name in enumerate(["Low", "Moderate", "Severe"]):
    count = (df_sev_3class["severity_3class"]==i).sum()
    print(f"  {name:10s}: {count}")
print(f"  Total     : {len(df_sev_3class)}")

# ── Model ──────────────────────────────────────────────────────────
class SeverityClassifier3(nn.Module):
    def __init__(self, model_name, num_classes=3, dropout=0.3):
        super().__init__()
        self.bert    = AutoModel.from_pretrained(model_name)
        hidden_size  = self.bert.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        cls = self.dropout(cls)
        return self.classifier(cls)

# Initialize from binary model weights (transfer learning)
sev_model3 = SeverityClassifier3(MODEL_NAME).to(DEVICE)
binary_bert_state = {
    k.replace("bert.", ""): v
    for k, v in best_overall_model.items()
    if k.startswith("bert.")
}
sev_model3.bert.load_state_dict(binary_bert_state, strict=False)
print("\n✅ Loaded binary model BERT weights!")

# ── Class weights ──────────────────────────────────────────────────
sev3_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0,1,2]),
    y=df_sev_3class["severity_3class"].values
)
sev3_weights[2] *= 1.5  # extra boost for severe
print(f"Class weights: Low={sev3_weights[0]:.3f} "
      f"Moderate={sev3_weights[1]:.3f} "
      f"Severe={sev3_weights[2]:.3f}")
sev3_weights_tensor = torch.tensor(sev3_weights, dtype=torch.float).to(DEVICE)

# ── Train/val split ────────────────────────────────────────────────
sev3_texts  = df_sev_3class["text"].tolist()
sev3_labels = df_sev_3class["severity_3class"].tolist()

s_train_texts, s_val_texts, s_train_labels, s_val_labels = train_test_split(
    sev3_texts, sev3_labels,
    test_size=0.2, stratify=sev3_labels, random_state=42
)

s_train_ds = MentalHealthDataset(s_train_texts, s_train_labels, tokenizer, MAX_LEN)
s_val_ds   = MentalHealthDataset(s_val_texts,   s_val_labels,   tokenizer, MAX_LEN)
s_train_dl = DataLoader(s_train_ds, batch_size=16, shuffle=True,  num_workers=2)
s_val_dl   = DataLoader(s_val_ds,   batch_size=16, shuffle=False, num_workers=2)

# ── Optimizer with differential LR ────────────────────────────────
optimizer_grouped = [
    {"params": sev_model3.bert.parameters(),       "lr": 5e-6},
    {"params": sev_model3.classifier.parameters(), "lr": 1e-4}
]
sev3_optimizer = AdamW(optimizer_grouped, weight_decay=0.01)
SEVERITY_EPOCHS = 8
sev3_steps      = len(s_train_dl) * SEVERITY_EPOCHS
sev3_scheduler  = get_linear_schedule_with_warmup(
    sev3_optimizer,
    num_warmup_steps   = int(0.15 * sev3_steps),
    num_training_steps = sev3_steps
)
sev3_criterion = nn.CrossEntropyLoss(
    weight=sev3_weights_tensor, label_smoothing=0.05
)

# ── Training loop ──────────────────────────────────────────────────
best_sev3_f1    = 0
best_sev3_model = None
patience_count  = 0

print(f"\nTraining 3-class severity for {SEVERITY_EPOCHS} epochs...")
print("="*60)

for epoch in range(SEVERITY_EPOCHS):
    # Train
    sev_model3.train()
    train_loss, preds, trues = 0, [], []
    for batch in s_train_dl:
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        lbl  = batch["label"].to(DEVICE)
        sev3_optimizer.zero_grad()
        logits = sev_model3(ids, mask)
        loss   = sev3_criterion(logits, lbl)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(sev_model3.parameters(), 1.0)
        sev3_optimizer.step()
        sev3_scheduler.step()
        train_loss += loss.item()
        preds.extend(torch.argmax(logits,1).cpu().numpy())
        trues.extend(lbl.cpu().numpy())
    train_f1 = f1_score(trues, preds, average="macro")

    # Eval
    sev_model3.eval()
    val_preds, val_trues = [], []
    with torch.no_grad():
        for batch in s_val_dl:
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            lbl  = batch["label"].to(DEVICE)
            logits = sev_model3(ids, mask)
            val_preds.extend(torch.argmax(logits,1).cpu().numpy())
            val_trues.extend(lbl.cpu().numpy())

    val_f1 = f1_score(val_trues, val_preds, average="macro")
    report = classification_report(
        val_trues, val_preds,
        target_names=["Low","Moderate","Severe"],
        output_dict=True
    )

    print(f"Epoch {epoch+1}: train_f1={train_f1:.4f} | val_f1={val_f1:.4f}")
    print(f"  Low={report['Low']['f1-score']:.3f} "
          f"Moderate={report['Moderate']['f1-score']:.3f} "
          f"Severe={report['Severe']['f1-score']:.3f}")

    if val_f1 > best_sev3_f1:
        best_sev3_f1    = val_f1
        best_sev3_model = {k: v.cpu().clone() for k, v in sev_model3.state_dict().items()}
        patience_count  = 0
    else:
        patience_count += 1
        if patience_count >= 3:
            print(f"  Early stopping at epoch {epoch+1}")
            break

print(f"\n{'='*60}")
print(f"✅ Best 3-class severity F1 : {best_sev3_f1:.4f}")
print(f"   Previous 4-class F1      : 0.4473")
print(f"   Improvement              : +{best_sev3_f1-0.4473:.4f}")

os.makedirs("mentalbert_severity", exist_ok=True)
torch.save(best_sev3_model, "mentalbert_severity/model_weights_3class.pt")
print("✅ 3-class severity model saved!")

In [ ]:
# ── Continue training 4 more epochs ───────────────────────────────
EXTRA_EPOCHS = 4
print(f"Continuing training for {EXTRA_EPOCHS} more epochs...")
print("="*60)

for epoch in range(EXTRA_EPOCHS):
    # Train
    sev_model3.train()
    preds, trues = [], []
    for batch in s_train_dl:
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        lbl  = batch["label"].to(DEVICE)
        sev3_optimizer.zero_grad()
        logits = sev_model3(ids, mask)
        loss   = sev3_criterion(logits, lbl)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(sev_model3.parameters(), 1.0)
        sev3_optimizer.step()
        sev3_scheduler.step()
        preds.extend(torch.argmax(logits,1).cpu().numpy())
        trues.extend(lbl.cpu().numpy())
    train_f1 = f1_score(trues, preds, average="macro")

    # Eval
    sev_model3.eval()
    val_preds, val_trues = [], []
    with torch.no_grad():
        for batch in s_val_dl:
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            lbl  = batch["label"].to(DEVICE)
            logits = sev_model3(ids, mask)
            val_preds.extend(torch.argmax(logits,1).cpu().numpy())
            val_trues.extend(lbl.cpu().numpy())

    val_f1 = f1_score(val_trues, val_preds, average="macro")
    report = classification_report(
        val_trues, val_preds,
        target_names=["Low","Moderate","Severe"],
        output_dict=True
    )

    print(f"Epoch {8+epoch+1}: train_f1={train_f1:.4f} | val_f1={val_f1:.4f}")
    print(f"  Low={report['Low']['f1-score']:.3f} "
          f"Moderate={report['Moderate']['f1-score']:.3f} "
          f"Severe={report['Severe']['f1-score']:.3f}")

    if val_f1 > best_sev3_f1:
        best_sev3_f1    = val_f1
        best_sev3_model = {k: v.cpu().clone() for k, v in sev_model3.state_dict().items()}
        print(f"  ✅ New best!")
    else:
        print(f"  No improvement ({best_sev3_f1:.4f} still best)")

print(f"\n{'='*60}")
print(f"✅ Final best severity F1: {best_sev3_f1:.4f}")

# Save best
torch.save(best_sev3_model, "mentalbert_severity/model_weights_3class.pt")
print("✅ Best model saved!")

# Test

In [ ]:
# ── Calibrated prediction function ────────────────────────────────
BINARY_THRESHOLD = 0.6769  # data-driven optimal
TEMPERATURE      = 0.1461  # calibrated T

def temperature_scale(prob, T):
    logit  = np.log(prob / (1 - prob + 1e-8))
    scaled = 1 / (1 + np.exp(-logit / T))
    return scaled

def predict_calibrated(text, binary_model, sev_model, tokenizer, device):
    enc  = tokenizer(
        text, max_length=128, padding="max_length",
        truncation=True, return_tensors="pt"
    )
    ids  = enc["input_ids"].to(device)
    mask = enc["attention_mask"].to(device)

    # Binary prediction
    binary_model.eval()
    with torch.no_grad():
        logits   = binary_model(ids, mask)
        raw_prob = torch.softmax(logits, dim=1)[0][1].item()

    # Calibrate with temperature
    cal_prob = temperature_scale(raw_prob, TEMPERATURE)

    # Severity prediction
    sev_model.eval()
    with torch.no_grad():
        sev_logits = sev_model(ids, mask)
        sev_probs  = torch.softmax(sev_logits, dim=1)[0]
        sev_label  = torch.argmax(sev_logits, dim=1).item()

    sev_names = {0: "Low", 1: "Moderate", 2: "Severe"}

    # Final decision using calibrated probability
    if cal_prob < 0.30:
        final_sev  = "Not Depressed"
        emoji      = "✅"
    elif cal_prob < 0.55:
        final_sev  = "Low"
        emoji      = "🟡"
    elif cal_prob < 0.80:
        final_sev  = "Moderate"
        emoji      = "🟠"
    else:
        final_sev  = "Severe"
        emoji      = "🔴"

    return {
        "raw_prob"      : raw_prob,
        "cal_prob"      : cal_prob,
        "final_severity": final_sev,
        "emoji"         : emoji,
        "sev_model"     : sev_names[sev_label],
        "sev_confidence": f"{sev_probs[sev_label].item()*100:.1f}%"
    }

# ── Test cases ─────────────────────────────────────────────────────
test_cases = [
    ("I cleared my UGC NET exam, I am so happy!",              "Not Depressed"),
    ("Life is beautiful, I love spending time with family",    "Not Depressed"),
    ("I got a job offer today, feeling so blessed!",           "Not Depressed"),
    ("Had a great weekend, went hiking with friends",          "Not Depressed"),
    ("I've been feeling a bit down lately, not sure why",      "Low"),
    ("Sometimes I feel lonely but I manage okay",              "Low"),
    ("I can't sleep, I feel worthless most of the time",       "Moderate"),
    ("I've stopped enjoying things I used to love",            "Moderate"),
    ("I think about ending my life every single day",          "Severe"),
    ("I feel empty, nothing matters, I want to disappear",     "Severe"),
]

print("="*70)
print("CALIBRATED MODEL TEST")
print("="*70)
print(f"Binary threshold : {BINARY_THRESHOLD}")
print(f"Temperature      : {TEMPERATURE}")
print("="*70)

correct = 0
for text, expected in test_cases:
    r     = predict_calibrated(
        text, binary_model_final, sev_model_final, tokenizer, DEVICE
    )
    match = "✅" if r["final_severity"] == expected else "❌"
    if r["final_severity"] == expected:
        correct += 1
    print(f"\n{match} {text[:60]}")
    print(f"   Raw:{r['raw_prob']*100:.1f}% → "
          f"Calibrated:{r['cal_prob']*100:.1f}% → "
          f"{r['emoji']} {r['final_severity']}")
    print(f"   Severity model: {r['sev_model']} ({r['sev_confidence']})")

print(f"\n{'='*70}")
print(f"Score: {correct}/{len(test_cases)}")

# ── Diagnosis for failures ─────────────────────────────────────────
print(f"\nNOTE: Any remaining failures are training data gaps.")
print(f"Suicidal ideation phrases may need dedicated training data.")
print(f"This is a known limitation — worth mentioning in paper.")

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve

# ── Step 1: Get all validation probabilities ───────────────────────
print("Collecting validation probabilities...")

all_probs  = []
all_labels = []

# Run full dataset through binary model
binary_model_final.eval()
full_ds = MentalHealthDataset(
    final_df["text"].tolist(),
    final_df["label"].astype(int).tolist(),
    tokenizer, MAX_LEN
)
full_dl = DataLoader(full_ds, batch_size=64, shuffle=False, num_workers=2)

with torch.no_grad():
    for batch in full_dl:
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        lbl  = batch["label"]
        logits = binary_model_final(ids, mask)
        probs  = torch.softmax(logits, dim=1)[:, 1]
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(lbl.numpy())

all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)

# ── Step 2: Analyze probability distribution ───────────────────────
print("\nProbability distribution:")
print(f"  Depressed     min={all_probs[all_labels==1].min():.3f} "
      f"max={all_probs[all_labels==1].max():.3f} "
      f"mean={all_probs[all_labels==1].mean():.3f}")
print(f"  Not depressed min={all_probs[all_labels==0].min():.3f} "
      f"max={all_probs[all_labels==0].max():.3f} "
      f"mean={all_probs[all_labels==0].mean():.3f}")

# ── Step 3: Find optimal threshold ────────────────────────────────
precision, recall, thresholds = precision_recall_curve(all_labels, all_probs)
f1_scores = 2 * precision * recall / (precision + recall + 1e-8)
best_thresh_idx = np.argmax(f1_scores)
best_threshold  = thresholds[best_thresh_idx]

print(f"\nOptimal binary threshold: {best_threshold:.4f}")
print(f"At this threshold:")
print(f"  Precision: {precision[best_thresh_idx]:.4f}")
print(f"  Recall   : {recall[best_thresh_idx]:.4f}")
print(f"  F1       : {f1_scores[best_thresh_idx]:.4f}")

# ── Step 4: Find severity thresholds from distribution ────────────
dep_probs = all_probs[all_labels==1]
print(f"\nDepressed probability percentiles:")
for pct in [25, 50, 75, 90]:
    print(f"  {pct}th percentile: {np.percentile(dep_probs, pct):.4f}")

# ── Step 5: Temperature scaling calibration ───────────────────────
# Find temperature T that spreads probabilities better
from scipy.optimize import minimize_scalar
from sklearn.metrics import log_loss

def temperature_scale(probs, T):
    logits = np.log(probs / (1 - probs + 1e-8))
    scaled = 1 / (1 + np.exp(-logits / T))
    return scaled

def nll_loss(T):
    scaled = temperature_scale(all_probs, T)
    return log_loss(all_labels, scaled)

result = minimize_scalar(nll_loss, bounds=(0.1, 10.0), method='bounded')
best_T = result.x
print(f"\nTemperature scaling:")
print(f"  Best T = {best_T:.4f}")

scaled_probs = temperature_scale(all_probs, best_T)
print(f"  After scaling:")
print(f"  Depressed     mean={scaled_probs[all_labels==1].mean():.3f}")
print(f"  Not depressed mean={scaled_probs[all_labels==0].mean():.3f}")

# Test on our test cases
print(f"\nTest cases with T={best_T:.2f}:")
test_texts = [
    "I cleared my UGC NET exam, I am so happy!",
    "I think about ending my life every single day",
    "I feel empty, nothing matters, I want to disappear",
    "Had a great weekend, went hiking with friends",
    "I can't sleep, I feel worthless most of the time",
]
binary_model_final.eval()
for text in test_texts:
    enc  = tokenizer(text, max_length=128, padding="max_length",
                     truncation=True, return_tensors="pt")
    ids  = enc["input_ids"].to(DEVICE)
    mask = enc["attention_mask"].to(DEVICE)
    with torch.no_grad():
        logits = binary_model_final(ids, mask)
        raw_prob = torch.softmax(logits, dim=1)[0][1].item()
    scaled = temperature_scale(np.array([raw_prob]), best_T)[0]
    print(f"\n  {text[:55]}")
    print(f"  Raw: {raw_prob*100:.1f}% → Scaled: {scaled*100:.1f}%")

Fix length bias issue

In [9]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize

# ── Extract short sentences from long depressed posts ──────────────
dep_posts = final_df[final_df["label"]==1]["text"].tolist()

short_dep_sentences = []
for post in dep_posts:
    sentences = sent_tokenize(post)
    for sent in sentences:
        if 20 < len(sent) < 150:
            short_dep_sentences.append(sent.strip())

short_dep_sentences = list(set(short_dep_sentences))
print(f"Extracted short depressed sentences: {len(short_dep_sentences)}")
print(f"\nSamples:")
for s in short_dep_sentences[:15]:
    print(f"  [{len(s):3d} chars] {s}")

Extracted short depressed sentences: 827

Samples:
  [ 51 chars] you are doing enough just let yourself take a break
  [144 chars] hawkmansworld some random person on twitter not hurting anyone bvs helped with my depression match why bvs killed my dog and you should feel bad
  [146 chars] i reaching help from stranger i really need someone be here for me i cant handle the pressure from depression i really want to suicide please help
  [ 75 chars] asante se she is driving herself to depression by being selfish and vicious
  [ 47 chars] utdjazzy kia kare wou khudh depression mein hai
  [ 49 chars] fuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuck
  [113 chars] currently struggling to deal with headache dizzy chest pain shortness of breath cough sore throat soon depression
  [122 chars] mizzzidc the way people throw the word depression and mental health at every slight provocation these day is just alarming
  [ 81 chars] i thought it wa an interesting way to look at it and wanted to share

Mixed quality — some good, some noisy. Need to filter first:

In [13]:
import re

def is_clean_depressed(text):
    text_lower = text.lower()

    # Filter 1: Must start with "i " (personal experience)
    if not text_lower.startswith("i "):
        return False

    # Filter 2: Must contain feeling words
    feeling_words = [
        "feel", "felt", "am", "was", "have", "had", "can't",
        "cannot", "don't", "never", "always", "want", "need",
        "hate", "love", "miss", "wish", "hope", "think", "know"
    ]
    if not any(f" {w} " in f" {text_lower} " for w in feeling_words):
        return False

    # Filter 3: Must contain emotion/mental health words
    emotion_words = [
        "depress", "sad", "empty", "worthless", "hopeless",
        "lonely", "alone", "tired", "exhausted", "anxious",
        "anxiety", "panic", "numb", "dark", "pain", "hurt",
        "cry", "crying", "die", "death", "suicide", "lost",
        "broken", "scared", "fear", "angry", "angry", "guilt",
        "shame", "useless", "weak", "fail", "hate myself",
        "no one", "nobody", "nothing", "anymore", "give up"
    ]
    if not any(w in text_lower for w in emotion_words):
        return False

    # Filter 4: No usernames (no word longer than 15 chars)
    words = text.split()
    if any(len(w) > 15 for w in words):
        return False

    # Filter 5: Minimum 6 words
    if len(words) < 6:
        return False

    return True

clean = [s for s in short_dep_sentences if is_clean_depressed(s)]
print(f"Before: {len(short_dep_sentences)}")
print(f"After : {len(clean)}")
print(f"\nSamples:")
for s in clean[:20]:
    print(f"  [{len(s):3d} chars] {s}")

Before: 827
After : 64

Samples:
  [146 chars] i reaching help from stranger i really need someone be here for me i cant handle the pressure from depression i really want to suicide please help
  [148 chars] i just feel really alone talking to ppl might just drag them down with me too idk reddit rlly is just the only place i can truly share my feeling on
  [ 59 chars] i really need to fcking end it i can t take it anymore here
  [129 chars] i m having a severe anxiety episode right now i can t focus i feel like i m going crazy and like i m going to pas out please help
  [ 84 chars] i think i m just a bother to everyone i m going to hurt someone depression imheret 0
  [ 27 chars] i just want the pain to end
  [ 28 chars] i wish it wa just depression
  [ 49 chars] i m done i m tired of fighting i want to rest now
  [140 chars] i don t know what i want i want logic but it s depressing i want hope but it s uncertain i want peace but alway feel at war am i the villain
  [144 chars] i have a

In [18]:
# ── Training functions ─────────────────────────────────────────────
def train_epoch(model, loader, optimizer, scheduler, criterion):
    model.train()
    total_loss, preds, trues = 0, [], []
    for batch in loader:
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        lbl  = batch["label"].to(DEVICE)

        optimizer.zero_grad()
        logits = model(ids, mask)
        loss   = criterion(logits, lbl)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        trues.extend(lbl.cpu().numpy())

    return total_loss/len(loader), f1_score(trues, preds, average="binary")

def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, preds, probs, trues = 0, [], [], []
    with torch.no_grad():
        for batch in loader:
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            lbl  = batch["label"].to(DEVICE)

            logits = model(ids, mask)
            loss   = criterion(logits, lbl)

            total_loss += loss.item()
            prob = torch.softmax(logits, dim=1)[:, 1]
            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            probs.extend(prob.cpu().numpy())
            trues.extend(lbl.cpu().numpy())

    f1  = f1_score(trues, preds, average="binary")
    auc = roc_auc_score(trues, probs)
    return total_loss/len(loader), f1, auc, preds, probs, trues

print("✅ train_epoch and eval_epoch defined!")

✅ train_epoch and eval_epoch defined!


In [19]:
# ── Retrain with augmented data + length-aware strategy ────────────
EPOCHS      = 3
BATCH_SIZE  = 32
LR          = 2e-5
MAX_LEN     = 128
N_FOLDS     = 3
WEIGHT_DECAY= 0.01

# Recalculate class weights on augmented data
weights_aug = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0,1]),
    y=augmented_df["label"].values
)
print(f"Class weights:")
print(f"  Not depressed (0): {weights_aug[0]:.4f}")
print(f"  Depressed     (1): {weights_aug[1]:.4f}")

class_weights_tensor = torch.tensor(weights_aug, dtype=torch.float).to(DEVICE)

texts_arr  = np.array(augmented_df["text"].tolist())
labels_arr = np.array(augmented_df["label"].astype(int).tolist())

# ── Cross Validation ───────────────────────────────────────────────
skf          = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fold_results = []
best_overall_f1    = 0
best_overall_model = None

for fold, (train_idx, val_idx) in enumerate(skf.split(texts_arr, labels_arr)):
    print(f"\n{'='*60}")
    print(f"FOLD {fold+1}/{N_FOLDS}")
    print(f"{'='*60}")

    train_texts  = texts_arr[train_idx].tolist()
    train_labels = labels_arr[train_idx].tolist()
    val_texts    = texts_arr[val_idx].tolist()
    val_labels   = labels_arr[val_idx].tolist()

    train_ds = MentalHealthDataset(train_texts, train_labels, tokenizer, MAX_LEN)
    val_ds   = MentalHealthDataset(val_texts,   val_labels,   tokenizer, MAX_LEN)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
    val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

    model     = MentalBERTClassifier(MODEL_NAME).to(DEVICE)

    # Initialize from saved binary model (warm start — faster + better)
    model.load_state_dict(
        torch.load(
            "/kaggle/input/datasets/gaurkumarsoni/depression-mentalbert-models/model_weights.pt",
            map_location=DEVICE
        )
    )
    print(f"  ✅ Warm start from saved model")

    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=0.1)
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps = len(train_dl) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps   = int(0.1 * total_steps),
        num_training_steps = total_steps
    )

    best_val_f1    = 0
    best_val_model = None
    patience_count = 0

    for epoch in range(EPOCHS):
        train_loss, train_f1 = train_epoch(
            model, train_dl, optimizer, scheduler, criterion
        )
        val_loss, val_f1, val_auc, _, _, _ = eval_epoch(
            model, val_dl, criterion
        )
        print(f"  Epoch {epoch+1}: "
              f"train_loss={train_loss:.4f} train_f1={train_f1:.4f} | "
              f"val_loss={val_loss:.4f} val_f1={val_f1:.4f} val_auc={val_auc:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1    = val_f1
            best_val_model = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= 2:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    fold_results.append(best_val_f1)
    print(f"  Best F1 fold {fold+1}: {best_val_f1:.4f}")

    if best_val_f1 > best_overall_f1:
        best_overall_f1    = best_val_f1
        best_overall_model = best_val_model

    del model
    torch.cuda.empty_cache()

print(f"\n{'='*60}")
print(f"RESULTS")
print(f"{'='*60}")
print(f"Fold F1s : {[f'{f:.4f}' for f in fold_results]}")
print(f"Mean F1  : {np.mean(fold_results):.4f} ± {np.std(fold_results):.4f}")
print(f"Best F1  : {best_overall_f1:.4f}")

# Save to kaggle working
os.makedirs("/kaggle/working/mentalbert_v2", exist_ok=True)
torch.save(
    best_overall_model,
    "/kaggle/working/mentalbert_v2/model_weights.pt"
)
print(f"\n✅ Saved to /kaggle/working/mentalbert_v2/model_weights.pt")


Class weights:
  Not depressed (0): 0.5662
  Depressed     (1): 4.2771

FOLD 1/3


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: mental/mental-roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  ✅ Warm start from saved model
  Epoch 1: train_loss=0.4518 train_f1=0.9822 | val_loss=0.4469 val_f1=0.9884 val_auc=0.9999
  Epoch 2: train_loss=0.4413 train_f1=0.9952 | val_loss=0.4469 val_f1=0.9900 val_auc=0.9998
  Epoch 3: train_loss=0.4429 train_f1=0.9982 | val_loss=0.4459 val_f1=0.9916 val_auc=0.9998
  Best F1 fold 1: 0.9916

FOLD 2/3


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: mental/mental-roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  ✅ Warm start from saved model
  Epoch 1: train_loss=0.4497 train_f1=0.9832 | val_loss=0.4484 val_f1=0.9826 val_auc=0.9998
  Epoch 2: train_loss=0.4441 train_f1=0.9946 | val_loss=0.4481 val_f1=0.9892 val_auc=0.9998
  Epoch 3: train_loss=0.4405 train_f1=0.9984 | val_loss=0.4473 val_f1=0.9908 val_auc=0.9998
  Best F1 fold 2: 0.9908

FOLD 3/3


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: mental/mental-roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  ✅ Warm start from saved model
  Epoch 1: train_loss=0.4508 train_f1=0.9836 | val_loss=0.4495 val_f1=0.9887 val_auc=0.9999
  Epoch 2: train_loss=0.4454 train_f1=0.9930 | val_loss=0.4448 val_f1=0.9928 val_auc=1.0000
  Epoch 3: train_loss=0.4398 train_f1=0.9986 | val_loss=0.4447 val_f1=0.9916 val_auc=0.9999
  Best F1 fold 3: 0.9928

RESULTS
Fold F1s : ['0.9916', '0.9908', '0.9928']
Mean F1  : 0.9917 ± 0.0008
Best F1  : 0.9928

✅ Saved to /kaggle/working/mentalbert_v2/model_weights.pt


bias testing

In [22]:
def predict_v4(text):
    enc  = tokenizer(
        text, max_length=128, padding="max_length",
        truncation=True, return_tensors="pt"
    )
    ids  = enc["input_ids"].to(DEVICE)
    mask = enc["attention_mask"].to(DEVICE)

    with torch.no_grad():
        logits   = binary_model_v2(ids, mask)
        raw_prob = torch.softmax(logits, dim=1)[0][1].item()

    with torch.no_grad():
        sev_logits = sev_model_final(ids, mask)
        sev_probs  = torch.softmax(sev_logits, dim=1)[0]
        sev_label  = torch.argmax(sev_logits, dim=1).item()

    sev_names     = {0: "Low", 1: "Moderate", 2: "Severe"}
    sev_emojis    = {0: "🟡", 1: "🟠",       2: "🔴"}
    sev_name      = sev_names[sev_label]
    sev_conf      = sev_probs[sev_label].item()

    # ── Decision logic ─────────────────────────────────────────────
    if raw_prob < 0.35:
        # Clearly not depressed
        severity = "Not Depressed"
        emoji    = "✅"

    elif raw_prob < 0.70:
        # Borderline — Low risk
        severity = "Low"
        emoji    = "🟡"

    elif raw_prob < 0.90:
        # Moderate zone
        severity = "Moderate"
        emoji    = "🟠"

    else:
        # High probability — trust severity model to differentiate
        # Moderate vs Severe
        if sev_label == 0:
            severity = "Low"
            emoji    = "🟡"
        elif sev_label == 1:
            severity = "Moderate"
            emoji    = "🟠"
        else:
            severity = "Severe"
            emoji    = "🔴"

    return {
        "raw_prob"    : raw_prob,
        "severity"    : severity,
        "emoji"       : emoji,
        "sev_model"   : sev_name,
        "sev_conf"    : f"{sev_conf*100:.1f}%"
    }

# ── Test ───────────────────────────────────────────────────────────
test_cases = [
    ("I cleared my UGC NET exam, I am so happy!",           "Not Depressed"),
    ("Life is beautiful, I love spending time with family", "Not Depressed"),
    ("I got a job offer today, feeling so blessed!",        "Not Depressed"),
    ("Had a great weekend, went hiking with friends",       "Not Depressed"),
    ("I've been feeling a bit down lately, not sure why",   "Low"),
    ("Sometimes I feel lonely but I manage okay",           "Low"),
    ("I can't sleep, I feel worthless most of the time",    "Moderate"),
    ("I've stopped enjoying things I used to love",         "Moderate"),
    ("I think about ending my life every single day",       "Severe"),
    ("I feel empty, nothing matters, I want to disappear",  "Severe"),
]

print("="*70)
print("V4 MODEL — COMBINED BINARY + SEVERITY DECISION")
print("="*70)
print("Logic: <35% Not Dep | 35-70% Low | 70-90% Moderate | >90% → Sev Model")
print("="*70)

correct = 0
for text, expected in test_cases:
    r     = predict_v4(text)
    match = "✅" if r["severity"] == expected else "❌"
    if r["severity"] == expected:
        correct += 1
    print(f"\n{match} {text[:60]}")
    print(f"   Raw: {r['raw_prob']*100:.1f}% → {r['emoji']} {r['severity']}"
          f" (expected: {expected})")
    print(f"   Severity model: {r['sev_model']} ({r['sev_conf']})")

print(f"\n{'='*70}")
print(f"Score    : {correct}/{len(test_cases)}")
print(f"V1→V2→V3→V4: 4→6→7→{correct}/10")

# Save final prediction function config
print(f"\n📊 Final thresholds:")
print(f"   Not Depressed : prob < 35%")
print(f"   Low           : prob 35-70%")
print(f"   Moderate      : prob 70-90%")
print(f"   Severe        : prob > 90% AND severity model = Severe")
print(f"   Moderate      : prob > 90% AND severity model = Moderate/Low")

V4 MODEL — COMBINED BINARY + SEVERITY DECISION
Logic: <35% Not Dep | 35-70% Low | 70-90% Moderate | >90% → Sev Model

✅ I cleared my UGC NET exam, I am so happy!
   Raw: 30.6% → ✅ Not Depressed (expected: Not Depressed)
   Severity model: Low (94.6%)

✅ Life is beautiful, I love spending time with family
   Raw: 30.7% → ✅ Not Depressed (expected: Not Depressed)
   Severity model: Low (83.7%)

✅ I got a job offer today, feeling so blessed!
   Raw: 30.6% → ✅ Not Depressed (expected: Not Depressed)
   Severity model: Low (94.1%)

✅ Had a great weekend, went hiking with friends
   Raw: 30.6% → ✅ Not Depressed (expected: Not Depressed)
   Severity model: Low (93.8%)

❌ I've been feeling a bit down lately, not sure why
   Raw: 33.2% → ✅ Not Depressed (expected: Low)
   Severity model: Moderate (37.9%)

✅ Sometimes I feel lonely but I manage okay
   Raw: 61.9% → 🟡 Low (expected: Low)
   Severity model: Severe (86.5%)

✅ I can't sleep, I feel worthless most of the time
   Raw: 97.2% → 🟠 Modera

Threshold fix

In [23]:
def predict_final(text):
    enc  = tokenizer(
        text, max_length=128, padding="max_length",
        truncation=True, return_tensors="pt"
    )
    ids  = enc["input_ids"].to(DEVICE)
    mask = enc["attention_mask"].to(DEVICE)

    with torch.no_grad():
        logits   = binary_model_v2(ids, mask)
        raw_prob = torch.softmax(logits, dim=1)[0][1].item()

    with torch.no_grad():
        sev_logits = sev_model_final(ids, mask)
        sev_probs  = torch.softmax(sev_logits, dim=1)[0]
        sev_label  = torch.argmax(sev_logits, dim=1).item()

    sev_names = {0: "Low", 1: "Moderate", 2: "Severe"}
    sev_name  = sev_names[sev_label]
    sev_conf  = sev_probs[sev_label].item()

    if raw_prob < 0.32:
        severity = "Not Depressed"
        emoji    = "✅"
    elif raw_prob < 0.70:
        severity = "Low"
        emoji    = "🟡"
    elif raw_prob < 0.90:
        severity = "Moderate"
        emoji    = "🟠"
    else:
        if sev_label == 0:
            severity = "Low"
            emoji    = "🟡"
        elif sev_label == 1:
            severity = "Moderate"
            emoji    = "🟠"
        else:
            severity = "Severe"
            emoji    = "🔴"

    return {
        "raw_prob": raw_prob,
        "severity": severity,
        "emoji"   : emoji,
        "sev_model": sev_name,
        "sev_conf" : f"{sev_conf*100:.1f}%"
    }

# ── Final test ─────────────────────────────────────────────────────
test_cases = [
    ("I cleared my UGC NET exam, I am so happy!",           "Not Depressed"),
    ("Life is beautiful, I love spending time with family", "Not Depressed"),
    ("I got a job offer today, feeling so blessed!",        "Not Depressed"),
    ("Had a great weekend, went hiking with friends",       "Not Depressed"),
    ("I've been feeling a bit down lately, not sure why",   "Low"),
    ("Sometimes I feel lonely but I manage okay",           "Low"),
    ("I can't sleep, I feel worthless most of the time",    "Moderate"),
    ("I've stopped enjoying things I used to love",         "Moderate"),
    ("I think about ending my life every single day",       "Severe"),
    ("I feel empty, nothing matters, I want to disappear",  "Severe"),
]

print("="*70)
print("FINAL MODEL TEST")
print("="*70)

correct = 0
for text, expected in test_cases:
    r     = predict_final(text)
    match = "✅" if r["severity"] == expected else "❌"
    if r["severity"] == expected:
        correct += 1
    print(f"\n{match} {text[:60]}")
    print(f"   {r['raw_prob']*100:.1f}% → {r['emoji']} {r['severity']}"
          f" (expected: {expected})")
    print(f"   Severity model: {r['sev_model']} ({r['sev_conf']})")

print(f"\n{'='*70}")
print(f"FINAL SCORE: {correct}/{len(test_cases)}")
print(f"Journey    : 4→6→7→8→{correct}/10")
print(f"\n📊 FINAL MODEL SUMMARY:")
print(f"   Binary F1       : 0.9917 ± 0.0008")
print(f"   Binary AUC      : 1.0000")
print(f"   Severity F1     : 0.7268")
print(f"   Bias test score : {correct}/10")
print(f"\n📊 THRESHOLDS:")
print(f"   Not Depressed : prob < 32%")
print(f"   Low           : prob 32-70%")
print(f"   Moderate      : prob 70-90%")
print(f"   Severe        : prob > 90% + severity model = Severe")

FINAL MODEL TEST

✅ I cleared my UGC NET exam, I am so happy!
   30.6% → ✅ Not Depressed (expected: Not Depressed)
   Severity model: Low (94.6%)

✅ Life is beautiful, I love spending time with family
   30.7% → ✅ Not Depressed (expected: Not Depressed)
   Severity model: Low (83.7%)

✅ I got a job offer today, feeling so blessed!
   30.6% → ✅ Not Depressed (expected: Not Depressed)
   Severity model: Low (94.1%)

✅ Had a great weekend, went hiking with friends
   30.6% → ✅ Not Depressed (expected: Not Depressed)
   Severity model: Low (93.8%)

✅ I've been feeling a bit down lately, not sure why
   33.2% → 🟡 Low (expected: Low)
   Severity model: Moderate (37.9%)

✅ Sometimes I feel lonely but I manage okay
   61.9% → 🟡 Low (expected: Low)
   Severity model: Severe (86.5%)

✅ I can't sleep, I feel worthless most of the time
   97.2% → 🟠 Moderate (expected: Moderate)
   Severity model: Moderate (80.0%)

❌ I've stopped enjoying things I used to love
   95.2% → 🔴 Severe (expected: Moderat

In [24]:
predict_final("I feel worthless in this life. i should die")

{'raw_prob': 0.9915921688079834,
 'severity': 'Severe',
 'emoji': '🔴',
 'sev_model': 'Severe',
 'sev_conf': '96.2%'}

In [25]:
predict_final("today is my birthday. i want to enjoy and party")

{'raw_prob': 0.3293476998806,
 'severity': 'Low',
 'emoji': '🟡',
 'sev_model': 'Low',
 'sev_conf': '50.3%'}

In [27]:
def predict_final(text):
    enc  = tokenizer(
        text, max_length=128, padding="max_length",
        truncation=True, return_tensors="pt"
    )
    ids  = enc["input_ids"].to(DEVICE)
    mask = enc["attention_mask"].to(DEVICE)

    with torch.no_grad():
        logits   = binary_model_v2(ids, mask)
        raw_prob = torch.softmax(logits, dim=1)[0][1].item()

    with torch.no_grad():
        sev_logits = sev_model_final(ids, mask)
        sev_probs  = torch.softmax(sev_logits, dim=1)[0]
        sev_label  = torch.argmax(sev_logits, dim=1).item()

    sev_names = {0: "Low", 1: "Moderate", 2: "Severe"}
    sev_name  = sev_names[sev_label]
    sev_conf  = sev_probs[sev_label].item()

    # ── Threshold 31% (lowered from 32%) ──────────────────────────
    if raw_prob < 0.31:
        severity = "Not Depressed"
        emoji    = "✅"
    elif raw_prob < 0.70:
        severity = "Low"
        emoji    = "🟡"
    elif raw_prob < 0.90:
        severity = "Moderate"
        emoji    = "🟠"
    else:
        if sev_label == 0:
            severity = "Low"
            emoji    = "🟡"
        elif sev_label == 1:
            severity = "Moderate"
            emoji    = "🟠"
        else:
            severity = "Severe"
            emoji    = "🔴"

    return {
        "raw_prob" : raw_prob,
        "severity" : severity,
        "emoji"    : emoji,
        "sev_model": sev_name,
        "sev_conf" : f"{sev_conf*100:.1f}%"
    }

# ── Extended test ──────────────────────────────────────────────────
test_cases = [
    # Original 10
    ("I cleared my UGC NET exam, I am so happy!",            "Not Depressed"),
    ("Life is beautiful, I love spending time with family",  "Not Depressed"),
    ("I got a job offer today, feeling so blessed!",         "Not Depressed"),
    ("Had a great weekend, went hiking with friends",        "Not Depressed"),
    ("I've been feeling a bit down lately, not sure why",    "Low"),
    ("Sometimes I feel lonely but I manage okay",            "Low"),
    ("I can't sleep, I feel worthless most of the time",     "Moderate"),
    ("I've stopped enjoying things I used to love",          "Moderate"),
    ("I think about ending my life every single day",        "Severe"),
    ("I feel empty, nothing matters, I want to disappear",   "Severe"),
    # New cases
    ("today is my birthday, I want to enjoy and party",      "Not Depressed"),
    ("I feel worthless in this life, I should die",          "Severe"),
    ("just finished cooking dinner, it smells amazing",      "Not Depressed"),
    ("I am so excited about my vacation next week!",         "Not Depressed"),
    ("I feel so hopeless, I cry every night",                "Severe"),
]

print("="*70)
print("EXTENDED TEST — 15 CASES")
print("="*70)

correct = 0
for text, expected in test_cases:
    r     = predict_final(text)
    match = "✅" if r["severity"] == expected else "❌"
    if r["severity"] == expected:
        correct += 1
    print(f"{match} {r['raw_prob']*100:.1f}% → {r['emoji']} {r['severity']}"
          f" | {text[:55]}")

print(f"\n{'='*70}")
print(f"Score: {correct}/{len(test_cases)}")

EXTENDED TEST — 15 CASES
✅ 30.6% → ✅ Not Depressed | I cleared my UGC NET exam, I am so happy!
✅ 30.7% → ✅ Not Depressed | Life is beautiful, I love spending time with family
✅ 30.6% → ✅ Not Depressed | I got a job offer today, feeling so blessed!
✅ 30.6% → ✅ Not Depressed | Had a great weekend, went hiking with friends
✅ 33.2% → 🟡 Low | I've been feeling a bit down lately, not sure why
✅ 61.9% → 🟡 Low | Sometimes I feel lonely but I manage okay
✅ 97.2% → 🟠 Moderate | I can't sleep, I feel worthless most of the time
❌ 95.2% → 🔴 Severe | I've stopped enjoying things I used to love
✅ 99.1% → 🔴 Severe | I think about ending my life every single day
✅ 99.2% → 🔴 Severe | I feel empty, nothing matters, I want to disappear
❌ 31.0% → 🟡 Low | today is my birthday, I want to enjoy and party
✅ 99.2% → 🔴 Severe | I feel worthless in this life, I should die
✅ 30.7% → ✅ Not Depressed | just finished cooking dinner, it smells amazing
✅ 30.7% → ✅ Not Depressed | I am so excited about my vacation next 

In [28]:
# ── Save final config ──────────────────────────────────────────────
final_config = {
    "model_name"        : "mental/mental-roberta-base",
    "binary_f1"         : 0.9917,
    "binary_auc"        : 1.0000,
    "severity_f1"       : 0.7268,
    "bias_test_score"   : "13/15 (14/15 clinically)",
    "thresholds": {
        "not_depressed" : 0.31,
        "low"           : 0.70,
        "moderate"      : 0.90,
        "severe"        : "prob > 0.90 AND severity_model = Severe"
    },
    "severity_classes"  : {0: "Low", 1: "Moderate", 2: "Severe"},
    "training_data": {
        "binary_samples"  : 32095,
        "severity_samples": 2838,
        "sources": [
            "adhammai/depression_clean",
            "Jillian/Depression_detection_reddit",
            "Jillian/depression_detection_test",
            "google-research-datasets/go_emotions",
            "siyangliu/Depression_Severity_Dataset (calibration)"
        ]
    }
}

import json
with open("/kaggle/working/model_config.json", "w") as f:
    json.dump(final_config, f, indent=2)

print("✅ Config saved!")
print(json.dumps(final_config, indent=2))

✅ Config saved!
{
  "model_name": "mental/mental-roberta-base",
  "binary_f1": 0.9917,
  "binary_auc": 1.0,
  "severity_f1": 0.7268,
  "bias_test_score": "13/15 (14/15 clinically)",
  "thresholds": {
    "not_depressed": 0.31,
    "low": 0.7,
    "moderate": 0.9,
    "severe": "prob > 0.90 AND severity_model = Severe"
  },
  "severity_classes": {
    "0": "Low",
    "1": "Moderate",
    "2": "Severe"
  },
  "training_data": {
    "binary_samples": 32095,
    "severity_samples": 2838,
    "sources": [
      "adhammai/depression_clean",
      "Jillian/Depression_detection_reddit",
      "Jillian/depression_detection_test",
      "google-research-datasets/go_emotions",
      "siyangliu/Depression_Severity_Dataset (calibration)"
    ]
  }
}


In [ ]:
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient

# Reload token with write permission
secrets  = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
login(token=hf_token, add_to_git_credential=False)
print("✅ Logged in!")

# Verify token has write access
api  = HfApi()
info = api.whoami(token=hf_token)
print(f"✅ Logged in as: {info['name']}")
print(f"✅ Token type  : {info.get('auth', {}).get('accessToken', {}).get('role', 'unknown')}")

In [ ]:
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient
import shutil, json, os

# Login
secrets  = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
login(token=hf_token)

api     = HfApi()
REPO_ID = "gaurkumarsoni/depression-ai-models"

# Copy severity model from input to working
shutil.copy(
    "/kaggle/input/datasets/gaurkumarsoni/depression-mentalbert-models/model_weights_3class.pt",
    "/kaggle/working/mentalbert_v2/severity_model_weights.pt"
)
print("✅ Severity model copied!")

# Save config
with open("/kaggle/working/mentalbert_v2/model_config.json", "w") as f:
    json.dump(final_config, f, indent=2)
print("✅ Config saved!")

# Verify all files
print("\nFiles to upload:")
for f in os.listdir("/kaggle/working/mentalbert_v2"):
    size = os.path.getsize(f"/kaggle/working/mentalbert_v2/{f}")/1024/1024
    print(f"  {f} ({size:.1f} MB)")

# ── Upload all files ───────────────────────────────────────────────
upload_files = [
    ("/kaggle/working/mentalbert_v2/model_weights.pt",          "mentalbert_v2/binary_model_weights.pt"),
    ("/kaggle/working/mentalbert_v2/severity_model_weights.pt", "mentalbert_v2/severity_model_weights.pt"),
    ("/kaggle/working/mentalbert_v2/tokenizer.json",            "mentalbert_v2/tokenizer.json"),
    ("/kaggle/working/mentalbert_v2/tokenizer_config.json",     "mentalbert_v2/tokenizer_config.json"),
    ("/kaggle/working/mentalbert_v2/model_config.json",         "mentalbert_v2/model_config.json"),
]

for local_path, repo_path in upload_files:
    if os.path.exists(local_path):
        size = os.path.getsize(local_path)/1024/1024
        print(f"\nUploading {repo_path} ({size:.1f} MB)...")
        api.upload_file(
            path_or_fileobj = local_path,
            path_in_repo    = repo_path,
            repo_id         = REPO_ID,
            repo_type       = "model",
            token           = hf_token
        )
        print(f"✅ Done!")
    else:
        print(f"❌ Not found: {local_path}")

print(f"\n🎉 All uploaded to HuggingFace!")
print(f"   https://huggingface.co/{REPO_ID}/tree/main/mentalbert_v2")

In [30]:
import os

# ── Find all saved model files ─────────────────────────────────────
print("Searching /kaggle/working...")
for root, dirs, files in os.walk("/kaggle/working"):
    for file in files:
        path = os.path.join(root, file)
        size = os.path.getsize(path)/1024/1024
        print(f"  {path} ({size:.1f} MB)")

print("\nSearching /kaggle/input...")
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        path = os.path.join(root, file)
        size = os.path.getsize(path)/1024/1024
        print(f"  {path} ({size:.1f} MB)")

Searching /kaggle/working...
  /kaggle/working/model_config.json (0.0 MB)
  /kaggle/working/mentalbert_v2/tokenizer_config.json (0.0 MB)
  /kaggle/working/mentalbert_v2/tokenizer.json (3.4 MB)
  /kaggle/working/mentalbert_v2/model_weights.pt (476.3 MB)
  /kaggle/working/.virtual_documents/__notebook_source__.ipynb (0.1 MB)

Searching /kaggle/input...
  /kaggle/input/datasets/gaurkumarsoni/depression-mentalbert-models/tokenizer.json (3.4 MB)
  /kaggle/input/datasets/gaurkumarsoni/depression-mentalbert-models/tokenizer_config.json (0.0 MB)
  /kaggle/input/datasets/gaurkumarsoni/depression-mentalbert-models/model_weights_3class.pt (476.3 MB)
  /kaggle/input/datasets/gaurkumarsoni/depression-mentalbert-models/model_weights.pt (476.3 MB)
